In [ ]:
pip install nest_asyncio

In [5]:
pip install praw


In [6]:
pip install asyncpraw

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.4/196.4 kB 5.6 MB/s eta 0:00:00


In [ ]:
import asyncpraw
import pandas as pd
from datetime import datetime
import asyncio
import nest_asyncio

# Apply nest_asyncio to allow nested event loops
nest_asyncio.apply()

# Step 1: Initialize Async Reddit API
async def collect_posts():

    reddit = asyncpraw.Reddit(
    client_id="W4mZQM0TOqxjVL9no_3ovw",
    client_secret="ydvwS7nI4di5dcDp8TJbsQ_RyZifYA",
    username="Unlucky-Mulberry-376",

    user_agent="Mah&Moh research"
    )

    # Step 2: Define Search Parameters
    search_term = "ChatGPT"
    start_date = int(datetime(2022, 11, 30).timestamp())  # Convert to Unix timestamp
    end_date = int(datetime.now().timestamp())

    # Step 3: Collect Posts
    posts = []
    subreddit = await reddit.subreddit("all")
    async for submission in subreddit.search(search_term, sort="new", syntax="lucene", time_filter="all"):
        post_date = submission.created_utc
        if start_date <= post_date <= end_date:
            posts.append({
                "id": submission.id,
                "title": submission.title,
                "text": submission.selftext,
                "score": submission.score,
                "num_comments": submission.num_comments,
                "url": submission.url,
                "created_utc": datetime.fromtimestamp(post_date).strftime('%Y-%m-%d %H:%M:%S')
            })

    # Step 4: Save Data to CSV
    df = pd.DataFrame(posts)
    df.to_csv("reddit_chatgpt_posts.csv", index=False)

    print(f"Collected {len(posts)} posts related to '{search_term}'.")

# Run the async function
await collect_posts()

Collected 100 posts related to 'ChatGPT'.


In [ ]:
import asyncpraw
import pandas as pd
from datetime import datetime
import asyncio
import nest_asyncio

# Apply nest_asyncio to allow nested event loops (for Jupyter Notebook)
nest_asyncio.apply()

# Step 1: Initialize Async Reddit API
async def collect_posts():
    reddit = asyncpraw.Reddit(
    client_id="W4mZQM0TOqxjVL9no_3ovw",
    client_secret="ydvwS7nI4di5dcDp8TJbsQ_RyZifYA",
    username="Unlucky-Mulberry-376",
    user_agent="Mah&Moh research"
    )

    # Test authentication
    print(f"Authenticated as: {await reddit.user.me()}")

    # Step 2: Define Search Parameters
    search_term = "ChatGPT"
    start_date = int(datetime(2022, 11, 30).timestamp())  # Convert to Unix timestamp
    end_date = int(datetime.now().timestamp())

    # Step 3: Collect Posts with Pagination
    posts = []
    subreddit = await reddit.subreddit("all")
    after = None  # Start with no "after" parameter

    while True:
        submissions = [submission async for submission in subreddit.search(
            search_term, sort="new", syntax="lucene", time_filter="all", limit=1000, params={"after": after}
        )]
        if not submissions:
            break  # Stop if no more posts are returned

        for submission in submissions:
            post_date = submission.created_utc
            if start_date <= post_date <= end_date:
                posts.append({
                    "id": submission.id,
                    "title": submission.title,
                    "text": submission.selftext,
                    "score": submission.score,
                    "num_comments": submission.num_comments,
                    "url": submission.url,
                    "created_utc": datetime.fromtimestamp(post_date).strftime('%Y-%m-%d %H:%M:%S')
                })

        # Update the "after" parameter for the next batch
        after = submissions[-1].fullname

        print(f"Collected {len(posts)} posts so far...")

        # Optional: Add a delay to avoid hitting rate limits
        await asyncio.sleep(2)

    # Step 4: Save Data to CSV
    df = pd.DataFrame(posts)
    df.to_csv("reddit_chatgpt_posts3.csv", index=False)

    print(f"Finished collecting {len(posts)} posts related to '{search_term}'.")

# Run the async function
await collect_posts()

Authenticated as: None
Collected 246 posts so far...
Finished collecting 246 posts related to 'ChatGPT'.


In [ ]:
import asyncpraw
import pandas as pd
from datetime import datetime
import asyncio
import nest_asyncio
import os

# Apply nest_asyncio to allow nested event loops (for Jupyter Notebook)
nest_asyncio.apply()

# Define the state file path
STATE_FILE = "last_processed_post.txt"

# Function to load the last processed post's fullname
def load_last_processed_post():
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE, "r") as f:
            return f.read().strip()
    return None

# Function to save the last processed post's fullname
def save_last_processed_post(fullname):
    with open(STATE_FILE, "w") as f:
        f.write(fullname)

# Step 1: Initialize Async Reddit API
async def collect_posts():
    reddit = asyncpraw.Reddit(
        client_id="W4mZQM0TOqxjVL9no_3ovw",
        client_secret="ydvwS7nI4di5dcDp8TJbsQ_RyZifYA",
        username="Unlucky-Mulberry-376",
        user_agent="Mah&Moh research"
    )

    # Test authentication
    print(f"Authenticated as: {await reddit.user.me()}")

    # Step 2: Define Search Parameters
    search_term = "ChatGPT"
    start_date = int(datetime(2022, 11, 30).timestamp())  # Convert to Unix timestamp
    end_date = int(datetime.now().timestamp())

    # Load the last processed post's fullname
    after = load_last_processed_post()

    # Step 3: Collect Posts with Pagination
    posts = []
    subreddit = await reddit.subreddit("all")
    daily_target = 1500  # Target number of posts per day

    while len(posts) < daily_target:
        submissions = [submission async for submission in subreddit.search(
            search_term, sort="new", syntax="lucene", time_filter="all", limit=100, params={"after": after}
        )]
        if not submissions:
            break  # Stop if no more posts are returned

        for submission in submissions:
            post_date = submission.created_utc
            if start_date <= post_date <= end_date:
                posts.append({
                    "id": submission.id,
                    "title": submission.title,
                    "text": submission.selftext,
                    "score": submission.score,
                    "num_comments": submission.num_comments,
                    "url": submission.url,
                    "created_utc": datetime.fromtimestamp(post_date).strftime('%Y-%m-%d %H:%M:%S')
                })

        # Update the "after" parameter for the next batch
        after = submissions[-1].fullname

        print(f"Collected {len(posts)} posts so far...")

        # Optional: Add a delay to avoid hitting rate limits
        await asyncio.sleep(2)

        # Stop if we've reached the daily target
        if len(posts) >= daily_target:
            break

    # Step 4: Save Data to CSV
    df = pd.DataFrame(posts)
    df.to_csv("reddit_chatgpt_posts4.csv", mode="a", header=not os.path.exists("reddit_chatgpt_posts.csv"), index=False)

    # Save the last processed post's fullname
    save_last_processed_post(after)

    print(f"Finished collecting {len(posts)} posts related to '{search_term}'.")

# Run the async function
await collect_posts()

Authenticated as: None
Collected 100 posts so far...
Collected 200 posts so far...
Collected 246 posts so far...
Finished collecting 246 posts related to 'ChatGPT'.


In [ ]:
pip install praw

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 12.0 MB/s eta 0:00:00


In [ ]:
import praw
import pandas as pd
from datetime import datetime

# Reddit API credentials (replace with your actual credentials)
REDDIT_CLIENT_ID = "your_client_id"
REDDIT_CLIENT_SECRET = "your_client_secret"
REDDIT_USER_AGENT = "your_user_agent"

# Initialize PRAW Reddit instance
reddit = praw.Reddit(
    client_id="W4mZQM0TOqxjVL9no_3ovw",
    client_secret="ydvwS7nI4di5dcDp8TJbsQ_RyZifYA",
    user_agent="Mah&Moh research"
)

# Define search parameters
search_term = "ChatGPT"
num_posts = 500  # Number of posts to retrieve

# Fetch posts
posts = []
for submission in reddit.subreddit("all").search(search_term, sort="new", limit=num_posts):
    posts.append({
        "id": submission.id,
        "title": submission.title,
        "text": submission.selftext if submission.selftext else "",
        "score": submission.score,
        "num_comments": submission.num_comments,
        "url": submission.url,
        "created_utc": datetime.fromtimestamp(submission.created_utc).strftime('%Y-%m-%d %H:%M:%S')
    })

# Save data to CSV
df = pd.DataFrame(posts)
df.to_csv("reddit_chatgpt_posts.csv", index=False)

print(f"Finished collecting {len(posts)} posts related to '{search_term}'.")


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Finished collecting 248 posts related to 'ChatGPT'.


In [ ]:
import praw
import pandas as pd
import time
from datetime import datetime, timedelta

# 🔹 Replace with your actual credentials
REDDIT_CLIENT_ID = "W4mZQM0TOqxjVL9no_3ovw"
REDDIT_CLIENT_SECRET = "ydvwS7nI4di5dcDp8TJbsQ_RyZifYA"
REDDIT_USER_AGENT = "Mah&Moh research"
REDDIT_USERNAME = "Unlucky-Mulberry-376"

# ✅ Authenticate with Reddit API
reddit = praw.Reddit(
    client_id=REDDIT_CLIENT_ID,
    client_secret=REDDIT_CLIENT_SECRET,
    user_agent=REDDIT_USER_AGENT,
    username=REDDIT_USERNAME,

)

# 🔍 Define search parameters
search_term = "ChatGPT"
subreddits = ["all", "technology", "artificial", "machinelearning"]  # Query multiple subreddits
days_per_chunk = 7  # Split search by week to collect more posts
start_date = datetime(2022, 11, 30)
end_date = datetime(2023, 10, 31)

# 📌 Collect posts in time chunks
posts = []
current_date = end_date

while current_date > start_date:
    after = int((current_date - timedelta(days=days_per_chunk)).timestamp())
    before = int(current_date.timestamp())

    for subreddit in subreddits:
        print(f"Fetching posts from r/{subreddit} from {current_date - timedelta(days=days_per_chunk)} to {current_date}")

        try:
            for submission in reddit.subreddit(subreddit).search(search_term, sort="new", limit=500):
                post_time = datetime.fromtimestamp(submission.created_utc)

                # Ensure posts are within the correct date range
                if start_date <= post_time <= end_date:
                    posts.append({
                        "id": submission.id,
                        "title": submission.title,
                        "text": submission.selftext if submission.selftext else "",
                        "score": submission.score,
                        "num_comments": submission.num_comments,
                        "url": submission.url,
                        "subreddit": subreddit,
                        "created_utc": post_time.strftime('%Y-%m-%d %H:%M:%S')
                    })

            print(f"✅ Collected {len(posts)} total posts so far...")

        except Exception as e:
            print(f"⚠️ Error fetching from r/{subreddit}: {e}")
            time.sleep(5)  # Pause before retrying to avoid rate limits

    # Move to the next time chunk
    current_date -= timedelta(days=days_per_chunk)
    time.sleep(2)  # Avoid API rate limits

# 📂 Save data to CSV
df = pd.DataFrame(posts)
df.to_csv("reddit_chatgpt_posts_large.csv", index=False)

print(f"✅ Finished collecting {len(posts)} posts related to '{search_term}' from multiple subreddits.")


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Fetching posts from r/all from 2023-10-24 00:00:00 to 2023-10-31 00:00:00


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



✅ Collected 0 total posts so far...
Fetching posts from r/technology from 2023-10-24 00:00:00 to 2023-10-31 00:00:00


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



...

✅ Collected 611 total posts so far...
Fetching posts from r/artificial from 2022-11-29 00:00:00 to 2022-12-06 00:00:00


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



✅ Collected 611 total posts so far...
Fetching posts from r/machinelearning from 2022-11-29 00:00:00 to 2022-12-06 00:00:00


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



✅ Collected 624 total posts so far...
✅ Finished collecting 624 posts related to 'ChatGPT' from multiple subreddits.


In [ ]:
import praw
import asyncpraw
import pandas as pd
import time
from datetime import datetime, timedelta

# 🔹 Replace with your actual credentials
REDDIT_CLIENT_ID = "W4mZQM0TOqxjVL9no_3ovw"
REDDIT_CLIENT_SECRET = "ydvwS7nI4di5dcDp8TJbsQ_RyZifYA"
REDDIT_USER_AGENT = "Mah&Moh research"
REDDIT_USERNAME = "Unlucky-Mulberry-376"

# ✅ Authenticate with Reddit API
reddit = asyncpraw.Reddit(
    client_id=REDDIT_CLIENT_ID,
    client_secret=REDDIT_CLIENT_SECRET,
    user_agent=REDDIT_USER_AGENT,
    username=REDDIT_USERNAME,

)

# 🔍 Define search parameters
search_term = "ChatGPT"
subreddits = [
    "ChatGPT", "OpenAI", "ArtificialInteligence", "MachineLearning", "technology",
    "Futurology", "PromptEngineering", "LanguageTechnology", "AIChatbots",
     "AIArt", "writing", "DeepLearning", "ChatGPT", "chat gpt", "Chat GPT", "chat-gpt", "GPT chat",
    "GPT", "GPT-3", "GPT-3.5", "GPT-4", "GPT3", "GPT4", "ChatGPT 3.5", "ChatGPT 4",
    "OpenAI chatbot", "AI chatbot", "language model", "LLM", "conversational AI",
    "text generator", "AI assistant", "writing AI", "coding AI",
    "ChapGPT", "ChaatGPT", "ChatGP", "ChatPGT", "Chat GBT", "ChatGtp",
    "OpenAI's model", "the new AI", "that AI chat thing",
    "ChatGPT Plus", "ChatGPT free", "GPT-4o", "GPT-3.5 Turbo",
    "vs Bard", "vs Claude", "vs Llama", "better than Bing AI", "like Jasper",
    "AI writer", "AI tutor", "AI coder", "AI brainstorming", "AI summarizer",
    "Chatbot", "The AI"
] # Query multiple subreddits
days_per_chunk = 7  # Split search by week to collect more posts
start_date = datetime(2022, 11, 30)
end_date = datetime(2023, 10, 31)

# 📌 Collect posts in time chunks
posts = []
current_date = end_date

while current_date > start_date:
    after = int((current_date - timedelta(days=days_per_chunk)).timestamp())
    before = int(current_date.timestamp())

    for subreddit in subreddits:
        print(f"Fetching posts from r/{subreddit} from {current_date - timedelta(days=days_per_chunk)} to {current_date}")

        try:
            for submission in reddit.subreddit(subreddit).search(search_term, sort="new", limit=500):
                post_time = datetime.fromtimestamp(submission.created_utc)

                # Ensure posts are within the correct date range
                if start_date <= post_time <= end_date:
                    posts.append({
                        "id": submission.id,
                        "title": submission.title,
                        "text": submission.selftext if submission.selftext else "",
                        "score": submission.score,
                        "num_comments": submission.num_comments,
                        "url": submission.url,
                        "subreddit": subreddit,
                        "created_utc": post_time.strftime('%Y-%m-%d %H:%M:%S')
                    })

            print(f"✅ Collected {len(posts)} total posts so far...")

        except Exception as e:
            print(f"⚠️ Error fetching from r/{subreddit}: {e}")
            time.sleep(5)  # Pause before retrying to avoid rate limits

    # Move to the next time chunk
    current_date -= timedelta(days=days_per_chunk)
    time.sleep(2)  # Avoid API rate limits

# 📂 Save data to CSV
df = pd.DataFrame(posts)
df.to_csv("reddit_chatgpt_posts_large2.csv", index=False)

print(f"✅ Finished collecting {len(posts)} posts related to '{search_term}' from multiple subreddits.")


Fetching posts from r/ChatGPT from 2023-10-24 00:00:00 to 2023-10-31 00:00:00
⚠️ Error fetching from r/ChatGPT: 'coroutine' object has no attribute 'search'


Exception ignored in: <coroutine object SubredditHelper.__call__ at 0x7f61b61954e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/google/colab/_variable_inspector.py", line 27, in run
KeyError: '__builtins__'
Exception ignored in: <coroutine object SubredditHelper.__call__ at 0x7f61b61954e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/google/colab/_variable_inspector.py", line 27, in run
KeyError: '__builtins__'


Fetching posts from r/OpenAI from 2023-10-24 00:00:00 to 2023-10-31 00:00:00
⚠️ Error fetching from r/OpenAI: 'coroutine' object has no attribute 'search'


Exception ignored in: <coroutine object SubredditHelper.__call__ at 0x7f61b6027d30>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/google/colab/_variable_inspector.py", line 27, in run
KeyError: '__builtins__'
Exception ignored in: <coroutine object SubredditHelper.__call__ at 0x7f61b6027d30>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/google/colab/_variable_inspector.py", line 27, in run
KeyError: '__builtins__'


Fetching posts from r/ArtificialInteligence from 2023-10-24 00:00:00 to 2023-10-31 00:00:00
⚠️ Error fetching from r/ArtificialInteligence: 'coroutine' object has no attribute 'search'


...

Fetching posts from r/GPT-3.5 Turbo from 2023-05-16 00:00:00 to 2023-05-23 00:00:00
⚠️ Error fetching from r/GPT-3.5 Turbo: 'coroutine' object has no attribute 'search'


Exception ignored in: <coroutine object SubredditHelper.__call__ at 0x7f61b6027d30>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/google/colab/_variable_inspector.py", line 27, in run
KeyError: '__builtins__'
Exception ignored in: <coroutine object SubredditHelper.__call__ at 0x7f61b6027d30>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/google/colab/_variable_inspector.py", line 27, in run
KeyError: '__builtins__'


Fetching posts from r/vs Bard from 2023-05-16 00:00:00 to 2023-05-23 00:00:00
⚠️ Error fetching from r/vs Bard: 'coroutine' object has no attribute 'search'


Exception ignored in: <coroutine object SubredditHelper.__call__ at 0x7f61c1a02200>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/google/colab/_variable_inspector.py", line 27, in run
KeyError: '__builtins__'
Exception ignored in: <coroutine object SubredditHelper.__call__ at 0x7f61c1a02200>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/google/colab/_variable_inspector.py", line 27, in run
KeyError: '__builtins__'


Fetching posts from r/vs Claude from 2023-05-16 00:00:00 to 2023-05-23 00:00:00
⚠️ Error fetching from r/vs Claude: 'coroutine' object has no attribute 'search'


In [ ]:
import praw
import pandas as pd
import time
from datetime import datetime, timedelta

# 🔹 Replace with your actual credentials
REDDIT_CLIENT_ID = "W4mZQM0TOqxjVL9no_3ovw"
REDDIT_CLIENT_SECRET = "ydvwS7nI4di5dcDp8TJbsQ_RyZifYA"
REDDIT_USER_AGENT = "Mah&Moh research"
REDDIT_USERNAME = "Unlucky-Mulberry-376"

# ✅ Authenticate with Reddit API
reddit = praw.Reddit(
    client_id=REDDIT_CLIENT_ID,
    client_secret=REDDIT_CLIENT_SECRET,
    user_agent=REDDIT_USER_AGENT,
    username=REDDIT_USERNAME,

)

# 🔍 Define search parameters
search_term = "ChatGPT"
subreddits = [
    "ChatGPT", "OpenAI", "artificial", "MachineLearning", "technology",
    "Futurology", "PromptEngineering", "LanguageTechnology", "AIChatbots",
    "learnprogramming", "SideProject", "AIArt", "writing", "AskReddit",
    "Jobs", "education"
] # Query multiple subreddits
days_per_chunk = 7  # Split search by week to collect more posts
start_date = datetime(2022, 11, 30)
end_date = datetime(2023, 10, 31)

# 📌 Collect posts in time chunks
posts = []
current_date = end_date

while current_date > start_date:
    after = int((current_date - timedelta(days=days_per_chunk)).timestamp())
    before = int(current_date.timestamp())

    for subreddit in subreddits:
        print(f"Fetching posts from r/{subreddit} from {current_date - timedelta(days=days_per_chunk)} to {current_date}")

        try:
            for submission in reddit.subreddit(subreddit).search(search_term, sort="new", limit=500):
                post_time = datetime.fromtimestamp(submission.created_utc)

                # Ensure posts are within the correct date range
                if start_date <= post_time <= end_date:
                    posts.append({
                        "id": submission.id,
                        "title": submission.title,
                        "text": submission.selftext if submission.selftext else "",
                        "score": submission.score,
                        "num_comments": submission.num_comments,
                        "url": submission.url,
                        "subreddit": subreddit,
                        "created_utc": post_time.strftime('%Y-%m-%d %H:%M:%S')
                    })

            print(f"✅ Collected {len(posts)} total posts so far...")

        except Exception as e:
            print(f"⚠️ Error fetching from r/{subreddit}: {e}")
            time.sleep(5)  # Pause before retrying to avoid rate limits

    # Move to the next time chunk
    current_date -= timedelta(days=days_per_chunk)
    time.sleep(2)  # Avoid API rate limits

# 📂 Save data to CSV
df = pd.DataFrame(posts)
df.to_csv("reddit_chatgpt_posts_large2022-2023.csv", index=False)

print(f"✅ Finished collecting {len(posts)} posts related to '{search_term}' from multiple subreddits.")


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Fetching posts from r/ChatGPT from 2023-10-24 00:00:00 to 2023-10-31 00:00:00


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



✅ Collected 0 total posts so far...
Fetching posts from r/OpenAI from 2023-10-24 00:00:00 to 2023-10-31 00:00:00


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



...

✅ Collected 12871 total posts so far...
Fetching posts from r/AskReddit from 2022-11-29 00:00:00 to 2022-12-06 00:00:00


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



✅ Collected 12871 total posts so far...
Fetching posts from r/Jobs from 2022-11-29 00:00:00 to 2022-12-06 00:00:00


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



✅ Collected 12873 total posts so far...
Fetching posts from r/education from 2022-11-29 00:00:00 to 2022-12-06 00:00:00
✅ Collected 12912 total posts so far...
✅ Finished collecting 12912 posts related to 'ChatGPT' from multiple subreddits.
